# ML Pipeline: KarmaLego Temporal Features to Flat Prediction Model

This notebook implements a straightforward prediction flow:

1. Build 48h observation-window features.
2. Build future complication labels.
3. Split patients into train/test.
4. Discover KarmaLego patterns on train only.
5. Apply those patterns to train and test.
6. Join optional QA/context features.
7. Train a simple baseline model per outcome.

This is intentionally pragmatic. The goal is to test whether KarmaLego temporal
features add useful signal to a flat model, not to over-optimize a thesis-main model.


## 0. Environment Setup

Run the install cell once per environment if needed. The notebook expects to be
launched from the repository root.


In [1]:
# Run once per environment if imports fail.
# !pip install -e . -q
# !pip install -r requirements.txt -q


In [2]:
import os
import gc
import json
import hashlib
import math
import warnings
import joblib
from pathlib import Path
from ast import literal_eval

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse as sp

from core.io import validate_input, build_or_load_mappings, preprocess_dataframe
from core.karmalego import KarmaLego, TIRP
from core.parallel_runner import ParallelRunner

try:
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.multiclass import OneVsRestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_recall_curve
except ImportError as exc:
    raise ImportError("Install ML dependencies first: pip install -r requirements.txt") from exc

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

try:
    import optuna
    HAS_OPTUNA = True
except Exception:
    HAS_OPTUNA = False

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 120)

print("Imports OK")
print("XGBoost available:", HAS_XGBOOST)
print("Optuna available:", HAS_OPTUNA)
print("SHAP available:", HAS_SHAP)


c:\Users\shaha\Work\Personal\KarmaLego\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK
XGBoost available: True
Optuna available: True
SHAP available: True


## 1. Configuration

Start with one outcome and a conservative KarmaLego configuration. Once the flow
works, set `RUN_ALL_OUTCOMES = True`.


In [3]:
# Paths
DATA_DIR = Path("data/input")
OUTPUT_DIR = Path("data/output/ml_pipeline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEMPORAL_DATA_PATH = DATA_DIR / "temporal_data.csv"
QA_DATA_PATH = DATA_DIR / "qa_data.csv"
CONTEXT_DATA_PATH = DATA_DIR / "context_data.csv"

# Notebook controls
QUICK_RUN = False          # Set True for a small smoke test.
MAX_PATIENTS = None        # e.g. 500 for testing; None for full data.
RANDOM_STATE = 42
TEST_SIZE = 0.25
RUN_ALL_OUTCOMES = True
# Feature blocks
USE_QA_DATA = False
USE_CONTEXT_DATA = True
USE_COUNT_MINMAX_FROM_TRAIN = False  # optional train-fitted normalized count features
USE_COMPACT_SPARSE_TIRP_CACHE = False

# Inclusion/exclusion criteria to reduce irrelevant compute.
# %_PATTERN% events carry QA-like signal only when QA is enabled.
_temporal_filters = ["Value NOT LIKE '%Steady%'"]
if not USE_QA_DATA:
    _temporal_filters.append("ConceptName NOT LIKE '%_PATTERN%'")

INCLUSION_EXCLUSION_CRITERIA = {
    "temporal": _temporal_filters,
    "context": [],
    "qa": [],
}

# Discovery subset size for pattern mining.
DISCOVERY_POSITIVE_PATIENTS = 100
DISCOVERY_NEGATIVE_PATIENTS = 100

# Prediction setup
OBS_HOURS = 48
OBS_DAYS = 2
LABEL_AFTER_OBS_WINDOW = True  # features use 0..OBS_HOURS; labels use > OBS_HOURS
LABEL_END_HOURS = None         # set a number for a finite prediction horizon
USE_HOSPITALIZATION_DAY_IF_AVAILABLE = True

# KarmaLego settings
KL_MIN_VER_SUPP = 0.10
KL_NUM_RELATIONS = 3
KL_EPSILON = pd.Timedelta(minutes=1)
KL_MAX_DISTANCE = pd.Timedelta(hours=4)
KL_MIN_LENGTH = 1
KL_MAX_LENGTH = 4
MIN_SHARED_PATTERN_VERTICAL_SUPPORT = 0.20
FILTERED_TIRP_VALUE_DTYPE = "float32"  # use float32 for LR; sparse cache keeps zeros implicit


# Model selection / tuning
MODEL_FAMILY = "linear_lr"  # sparse multi-label linear baseline
ENABLE_FEATURE_PRUNING = False
GAIN_KEEP_FRACTION = 0.95
MIN_FEATURES_AFTER_PRUNING = 25
ENABLE_OPTUNA = False
OPTUNA_TRIALS = 30
OPTUNA_TIMEOUT_SECONDS = None


# Interpretation
ENABLE_SHAP = False
SHAP_MAX_SAMPLES = 500
SHAP_TOP_N = 30

# Parallelism
N_WORKERS = 1              # Use 1 first. Increase only after the flow is stable.
BATCH_SIZE = 100           # Used when N_WORKERS > 1.
APPLY_N_JOBS = N_WORKERS
SHARED_APPLY_CHUNK_SIZE = 1000

OUTCOMES = [
    "DISGLYCEMIA_EVENT_Hyperglycemia",
    "DISGLYCEMIA_EVENT_Hypoglycemia",
    "HYPEROSMOLALITY_EVENT",
    "CARDIO-VASCULAR_DISORDER_EVENT",
    "KIDNEY_COMPLICATION_EVENT",
    "DEATH_EVENT",
]

OUTCOMES_TO_RUN = OUTCOMES
OUTCOMES_TO_RUN


['DISGLYCEMIA_EVENT_Hyperglycemia',
 'DISGLYCEMIA_EVENT_Hypoglycemia',
 'HYPEROSMOLALITY_EVENT',
 'CARDIO-VASCULAR_DISORDER_EVENT',
 'KIDNEY_COMPLICATION_EVENT',
 'DEATH_EVENT']

## 2. Utility Functions


In [4]:
def read_csv_limited(path, max_patients=None):
    df = pd.read_csv(path)
    if max_patients is not None and "PatientId" in df.columns:
        keep = df["PatientId"].drop_duplicates().head(max_patients)
        df = df[df["PatientId"].isin(keep)].copy()
    return df


def parse_time_series(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        return pd.to_datetime(series)
    parsed = pd.to_datetime(series, errors="coerce")
    if parsed.isna().all():
        numeric = pd.to_numeric(series, errors="coerce")
        if numeric.isna().any():
            raise ValueError("Time column is neither parseable datetime nor numeric.")
        return numeric
    if parsed.isna().any():
        bad = series[parsed.isna()].head(5)
        raise ValueError(f"Partially unparseable datetime column. Examples:\n{bad}")
    return parsed


def add_time_columns(df):
    out = df.copy()
    out["_start"] = parse_time_series(out["StartDateTime"])
    out["_end"] = parse_time_series(out["EndDateTime"])
    return out


def build_admission_anchors(df):
    if "ConceptName" in df.columns:
        admission = df[df["ConceptName"].astype(str).eq("ADMISSION_EVENT")]
    else:
        admission = pd.DataFrame()

    if not admission.empty:
        anchors = admission.groupby("PatientId")["_start"].min()
    else:
        anchors = df.groupby("PatientId")["_start"].min()
    return anchors.rename("_admission_time")


def build_admission_lengths(df):
    if "PatientId" not in df.columns:
        raise ValueError("PatientId column is required to compute admission lengths.")
    if "_start" not in df.columns or "_end" not in df.columns:
        raise ValueError("Call add_time_columns() before computing admission lengths.")

    agg = df.groupby("PatientId").agg(
        _admission_start=("_start", "min"),
        _admission_end=("_end", "max"),
    )
    lengths = agg["_admission_end"] - agg["_admission_start"]
    if np.issubdtype(lengths.dtype, np.timedelta64):
        lengths = lengths.dt.total_seconds() / 3600.0
    else:
        lengths = pd.to_numeric(lengths, errors="coerce")
    return lengths.rename("_admission_length_hours")


def add_hours_since_admission(df, admission_anchors):
    out = df.copy()
    anchors = pd.Series(admission_anchors).rename("_admission_time")
    out = out.join(anchors, on="PatientId")
    if out["_admission_time"].isna().any():
        missing = out.loc[out["_admission_time"].isna(), "PatientId"].drop_duplicates().head(5).tolist()
        raise ValueError(f"Missing admission anchors for patients: {missing}")

    out["_hours_since_admission_start"] = (
        out["_start"] - out["_admission_time"]
    ).dt.total_seconds() / 3600.0
    out["_hours_since_admission_end"] = (
        out["_end"] - out["_admission_time"]
    ).dt.total_seconds() / 3600.0
    return out


def build_label_table(df, outcomes, obs_hours, label_after_obs=True, label_end_hours=None):
    if "PatientId" not in df.columns:
        raise ValueError("PatientId column is required to build labels.")
    if "ConceptName" not in df.columns:
        raise ValueError("ConceptName column is required to build labels.")
    if "_hours_since_admission_start" not in df.columns:
        raise ValueError("Call add_hours_since_admission() before building labels.")

    records = []
    first_onsets = []

    for pid, group in df.groupby("PatientId"):
        row = {"PatientId": pid}
        onset_row = {"PatientId": pid}
        group_concepts = group["ConceptName"].astype(str)
        group_values = group["Value"].astype(str) if "Value" in group.columns else pd.Series("", index=group.index)
        hours = group["_hours_since_admission_start"]
        if label_after_obs:
            time_mask = hours > obs_hours
        else:
            time_mask = hours.between(0, obs_hours, inclusive="both")
        if label_end_hours is not None:
            time_mask &= hours <= label_end_hours

        for outcome in outcomes:
            if outcome == "DISGLYCEMIA_EVENT_Hyperglycemia":
                mask = group_concepts.eq("DISGLYCEMIA_EVENT") & group_values.eq("Hyperglycemia")
            elif outcome == "DISGLYCEMIA_EVENT_Hypoglycemia":
                mask = group_concepts.eq("DISGLYCEMIA_EVENT") & group_values.eq("Hypoglycemia")
            else:
                mask = group_concepts.eq(outcome)
            mask &= time_mask
            onset_hours = hours.loc[mask].min()
            has_event = pd.notna(onset_hours)
            row[outcome] = int(has_event)
            onset_row[outcome] = float(onset_hours) if has_event else np.nan
        records.append(row)
        first_onsets.append(onset_row)

    labels = pd.DataFrame.from_records(records).set_index("PatientId")
    onsets = pd.DataFrame.from_records(first_onsets).set_index("PatientId")
    return labels, onsets


def make_observation_temporal_df(df, obs_hours, obs_days=None):
    if "_hours_since_admission_start" not in df.columns:
        raise ValueError("Call add_hours_since_admission() before building observation windows.")
    out = df.copy()
    out = out[out["_hours_since_admission_start"].between(0, obs_hours, inclusive="both")].copy()
    return out


def observation_event_counts(observation_temporal):
    if observation_temporal.empty:
        return pd.Series(dtype=float, name="observation_event_count")
    counts = observation_temporal.groupby("PatientId").size().astype(float)
    counts.name = "observation_event_count"
    return counts


def _match_sql_like(series, pattern):
    pattern = str(pattern).strip()
    if pattern.startswith("WHERE "):
        pattern = pattern[6:].strip()
    col = None
    if " NOT LIKE " in pattern:
        col, rhs = pattern.split(" NOT LIKE ", 1)
        negate = True
    elif " LIKE " in pattern:
        col, rhs = pattern.split(" LIKE ", 1)
        negate = False
    elif " != " in pattern:
        col, rhs = pattern.split(" != ", 1)
        negate = True
    elif " = " in pattern:
        col, rhs = pattern.split(" = ", 1)
        negate = False
    else:
        raise ValueError(f"Unsupported filter clause: {pattern}")

    col = col.strip()
    rhs = rhs.strip().strip("'").strip('"')
    if rhs.startswith("%") and rhs.endswith("%"):
        needle = rhs[1:-1]
        mask = series.astype(str).str.contains(needle, na=False, regex=False)
        return ~mask if negate else mask
    if rhs.startswith("%"):
        needle = rhs[1:]
        mask = series.astype(str).str.endswith(needle, na=False)
        return ~mask if negate else mask
    if rhs.endswith("%"):
        needle = rhs[:-1]
        mask = series.astype(str).str.startswith(needle, na=False)
        return ~mask if negate else mask
    if negate:
        return series.astype(str) != rhs
    return series.astype(str) == rhs


def apply_inclusion_exclusion_criteria(df, clauses):
    out = df.copy()
    for clause in clauses or []:
        if not clause:
            continue
        clause = str(clause).strip()
        if clause.startswith("WHERE "):
            clause = clause[6:].strip()
        if " NOT LIKE " in clause or " LIKE " in clause:
            if " NOT LIKE " in clause:
                col, rhs = clause.split(" NOT LIKE ", 1)
                negate = True
            else:
                col, rhs = clause.split(" LIKE ", 1)
                negate = False
            col = col.strip()
            rhs = rhs.strip().strip("'").strip('"')
            if rhs.startswith("%") and rhs.endswith("%"):
                needle = rhs[1:-1]
                mask = out[col].astype(str).str.contains(needle, na=False, regex=False)
                out = out[~mask] if negate else out[mask]
            elif rhs.startswith("%"):
                needle = rhs[1:]
                mask = out[col].astype(str).str.endswith(needle, na=False)
                out = out[~mask] if negate else out[mask]
            elif rhs.endswith("%"):
                needle = rhs[:-1]
                mask = out[col].astype(str).str.startswith(needle, na=False)
                out = out[~mask] if negate else out[mask]
            else:
                mask = out[col].astype(str) == rhs
                out = out[~mask] if negate else out[mask]
        elif " != " in clause:
            col, rhs = clause.split(" != ", 1)
            col = col.strip()
            rhs = rhs.strip().strip("'").strip('"')
            out = out[out[col].astype(str) != rhs]
        elif " = " in clause:
            col, rhs = clause.split(" = ", 1)
            col = col.strip()
            rhs = rhs.strip().strip("'").strip('"')
            out = out[out[col].astype(str) == rhs]
        else:
            raise ValueError(f"Unsupported filter clause: {clause}")
    return out


def make_stratify_key(y, obs_counts, max_bins=4):
    y = pd.Series(y).copy()
    counts = pd.Series(obs_counts).reindex(y.index)
    if counts.dropna().empty or counts.nunique(dropna=True) <= 1:
        return y.astype(str)

    counts = counts.fillna(counts.median())
    n_bins = min(int(max_bins), max(1, len(y) // 2))
    if n_bins <= 1:
        return y.astype(str)

    try:
        bins = pd.qcut(counts.rank(method="first"), q=n_bins, duplicates="drop")
        if getattr(bins, "nunique", lambda: 0)() <= 1 or bins.value_counts().min() < 2:
            return y.astype(str)
        return y.astype(str) + "::" + bins.astype(str)
    except Exception:
        return y.astype(str)


def _sample_with_length_stratification(patient_ids, lengths, target_n, random_state):
    patient_ids = list(pd.Index(patient_ids))
    target_n = int(min(max(target_n, 0), len(patient_ids)))
    if target_n <= 0:
        return []
    if target_n >= len(patient_ids):
        return patient_ids

    lengths = pd.Series(lengths).reindex(patient_ids)
    lengths = lengths.fillna(lengths.median() if not lengths.dropna().empty else 0)
    if lengths.nunique(dropna=True) <= 1 or len(patient_ids) < 4:
        return list(pd.Index(patient_ids).to_series().sample(n=target_n, random_state=random_state).tolist())

    n_bins = min(4, max(2, len(patient_ids) // 2))
    try:
        bins = pd.qcut(lengths.rank(method="first"), q=n_bins, duplicates="drop")
        if bins.nunique(dropna=True) <= 1 or bins.value_counts().min() < 2:
            raise ValueError("Not enough samples per bin for stratified sampling")
        sampled, _ = train_test_split(
            patient_ids,
            train_size=target_n,
            random_state=random_state,
            stratify=bins,
        )
        return list(sampled)
    except Exception:
        return list(pd.Index(patient_ids).to_series().sample(n=target_n, random_state=random_state).tolist())


def sample_discovery_patients(patient_ids, y, admission_lengths, positive_n, negative_n, random_state=RANDOM_STATE):
    patient_index = pd.Index(patient_ids)
    y = pd.Series(y).reindex(patient_index).dropna().astype(int)
    if y.empty:
        return []

    lengths = pd.Series(admission_lengths).reindex(patient_index)
    lengths = lengths.fillna(lengths.median() if not lengths.dropna().empty else 0)
    positive_n = int(max(0, positive_n))
    negative_n = int(max(0, negative_n))
    target_n = min(positive_n + negative_n, len(y))
    if target_n <= 0:
        return []

    groups = sorted(y.unique().tolist())
    if len(groups) == 1:
        sampled = _sample_with_length_stratification(y.index.to_numpy(), lengths, target_n, random_state)
        return [pid for pid in patient_index if pid in set(sampled)]

    target_by_group = {0: negative_n, 1: positive_n}
    selected = []
    for group in groups:
        group_ids = y[y == group].index.to_numpy()
        group_target = min(target_by_group.get(int(group), target_n // max(1, len(groups))), len(group_ids))
        sampled = _sample_with_length_stratification(
            group_ids,
            lengths.reindex(group_ids),
            group_target,
            random_state + int(group),
        )
        selected.extend(sampled)

    selected = [pid for pid in patient_index if pid in set(selected)]
    return selected[:target_n]


def split_for_outcome(labels, obs_counts, outcome):
    y_raw = labels[outcome].dropna().astype(int)
    candidates = [make_stratify_key(y_raw, obs_counts), y_raw.astype(str), None]
    last_error = None
    for stratify in candidates:
        try:
            train_ids, test_ids = train_test_split(
                y_raw.index.to_numpy(),
                test_size=TEST_SIZE,
                random_state=RANDOM_STATE,
                stratify=stratify,
            )
            return list(train_ids), list(test_ids), y_raw
        except ValueError as exc:
            last_error = exc
    raise last_error


def make_multilabel_stratify_key(Y, obs_counts=None):
    Y = Y.astype(int)
    combo = Y.astype(str).agg("".join, axis=1)
    counts = combo.value_counts()
    combo = combo.where(combo.map(counts) >= 2, "rare")
    if combo.value_counts().min() >= 2 and combo.nunique() > 1:
        return combo
    any_positive = Y.any(axis=1).astype(int).astype(str)
    if any_positive.value_counts().min() >= 2 and any_positive.nunique() > 1:
        return any_positive
    return None


def split_for_multilabel(labels, obs_counts, outcomes):
    Y = labels[outcomes].dropna().astype(int)
    candidates = [make_multilabel_stratify_key(Y, obs_counts), None]
    last_error = None
    for stratify in candidates:
        try:
            train_ids, test_ids = train_test_split(
                Y.index.to_numpy(),
                test_size=TEST_SIZE,
                random_state=RANDOM_STATE,
                stratify=stratify,
            )
            return list(train_ids), list(test_ids), Y
        except ValueError as exc:
            last_error = exc
    raise last_error


def filter_known_symbols(df, symbol_map):
    raw = df["ConceptName"].astype(str) + ":" + df["Value"].astype(str)
    return df[raw.isin(symbol_map)].copy()


def to_entity_list_for_patients(preprocessed_df, patient_ids):
    grouped = {pid: group for pid, group in preprocessed_df.groupby("PatientId")}
    entity_list = []
    for pid in patient_ids:
        group = grouped.get(pid)
        if group is None:
            entity_list.append([])
            continue
        group = group.sort_values(by=["StartDateTime", "EndDateTime"])
        entity_list.append(list(zip(group["StartDateTime"], group["EndDateTime"], group["symbol"])))
    return entity_list, list(patient_ids)


## 3. Load Data and Build Labels


In [5]:
raw_temporal = read_csv_limited(TEMPORAL_DATA_PATH, MAX_PATIENTS if QUICK_RUN else None)
validate_input(raw_temporal)

raw_temporal = add_time_columns(raw_temporal)
admission_anchors = build_admission_anchors(raw_temporal)
admission_lengths = build_admission_lengths(raw_temporal)
temporal_with_hours = add_hours_since_admission(raw_temporal, admission_anchors)
labels, first_onsets = build_label_table(temporal_with_hours, OUTCOMES, OBS_HOURS, LABEL_AFTER_OBS_WINDOW, LABEL_END_HOURS)

feature_temporal = apply_inclusion_exclusion_criteria(raw_temporal, INCLUSION_EXCLUSION_CRITERIA["temporal"])
feature_temporal = add_hours_since_admission(feature_temporal, admission_anchors)
SHARED_MAPPING_DIR = OUTPUT_DIR / "shared_mappings"
SHARED_MAPPING_DIR.mkdir(parents=True, exist_ok=True)
shared_symbol_map, shared_inverse_symbol_map = build_or_load_mappings(
    feature_temporal,
    mapping_dir=str(SHARED_MAPPING_DIR),
    reuse=False,
)
observation_temporal = make_observation_temporal_df(feature_temporal, OBS_HOURS, OBS_DAYS)
obs_counts = observation_event_counts(observation_temporal)

print("Raw temporal rows:", len(raw_temporal))
print("Observation rows:", len(observation_temporal))
print("Patients:", labels.shape[0])
label_counts = labels.sum().rename("positives").to_frame()
label_counts["prevalence"] = label_counts["positives"] / len(labels)
display(label_counts)

print("Admission length stats (hours):", admission_lengths.describe())


Raw temporal rows: 16799449
Observation rows: 6346443
Patients: 57078


,positives,prevalence
DISGLYCEMIA_EVENT_Hyperglycemia,15044,0.263569
DISGLYCEMIA_EVENT_Hypoglycemia,3202,0.056099
HYPEROSMOLALITY_EVENT,16918,0.296401
CARDIO-VASCULAR_DISORDER_EVENT,1185,0.020761
KIDNEY_COMPLICATION_EVENT,13697,0.239970
DEATH_EVENT,7244,0.126914


Admission length stats (hours): count    57078.000000
mean       172.269581
std        104.224168
min         48.000278
25%        102.750278
50%        149.891944
75%        215.883611
max       1037.783611
Name: _admission_length_hours, dtype: float64


## 4. Pattern Discovery and Application Helpers

Patterns are discovered on train patients only and then applied to both train and
test patients.


In [6]:
def kl_params(task_dir):
    return {
        "epsilon": KL_EPSILON,
        "max_distance": KL_MAX_DISTANCE,
        "min_ver_supp": KL_MIN_VER_SUPP,
        "num_relations": KL_NUM_RELATIONS,
        "min_length": KL_MIN_LENGTH,
        "max_length": KL_MAX_LENGTH,
        "inverse_mapping_path": str(SHARED_MAPPING_DIR / "inverse_symbol_map.json"),
        "show_progress": True,
    }


def load_patterns_dataframe(patterns_path):
    if not patterns_path.exists():
        return pd.DataFrame()
    df = pd.read_csv(patterns_path)
    if df.empty:
        return df
    if "symbols" in df.columns:
        df["symbols"] = df["symbols"].apply(lambda x: literal_eval(x) if isinstance(x, str) else x)
    if "relations" in df.columns:
        df["relations"] = df["relations"].apply(lambda x: literal_eval(x) if isinstance(x, str) else x)
    return df


def discover_or_load_patterns_for_outcome(train_obs_df, train_patient_ids, train_y, train_lengths, task_dir, outcome):
    task_dir.mkdir(parents=True, exist_ok=True)
    patterns_path = task_dir / "discovered_patterns.csv"
    discovery_ids_path = task_dir / "discovery_patient_ids.csv"

    if patterns_path.exists() and patterns_path.stat().st_size > 0:
        print(f"Loading cached patterns for {outcome}")
        return load_patterns_dataframe(patterns_path)

    train_pre = preprocess_dataframe(train_obs_df, shared_symbol_map)
    discovery_ids = sample_discovery_patients(
        train_patient_ids,
        train_y,
        train_lengths,
        DISCOVERY_POSITIVE_PATIENTS,
        DISCOVERY_NEGATIVE_PATIENTS,
    )
    discovery_df = train_pre[train_pre["PatientId"].isin(discovery_ids)].copy()
    discovery_entities, _ = to_entity_list_for_patients(discovery_df, discovery_ids)

    pd.Series(discovery_ids, name="PatientId").to_csv(discovery_ids_path, index=False)
    params = kl_params(task_dir)
    if N_WORKERS and N_WORKERS > 1:
        runner = ParallelRunner(max_workers=N_WORKERS)
        patterns_df = runner.parallel_batches(
            discovery_entities,
            params=params,
            batch_size=BATCH_SIZE,
            show_progress=True,
        )
    else:
        kl = KarmaLego(
            epsilon=KL_EPSILON,
            max_distance=KL_MAX_DISTANCE,
            min_ver_supp=KL_MIN_VER_SUPP,
            num_relations=KL_NUM_RELATIONS,
        )
        patterns_df = kl.discover_patterns(
            discovery_entities,
            min_length=KL_MIN_LENGTH,
            max_length=KL_MAX_LENGTH,
            inverse_mapping_path=str(SHARED_MAPPING_DIR / "inverse_symbol_map.json"),
            show_progress=True,
        )

    patterns_df.to_csv(patterns_path, index=False)

    # Drop temporary discovery memory before the next outcome starts.
    del train_pre, discovery_df, discovery_entities, patterns_df
    gc.collect()
    return load_patterns_dataframe(patterns_path)


def update_combined_pattern_bank(merged, outcome, patterns_df):
    if patterns_df is None or patterns_df.empty:
        return merged
    for row in patterns_df.itertuples(index=False):
        sig = pattern_signature_from_row(row)
        rec = merged.setdefault(sig, {
            "symbols": list(sig[0]),
            "relations": list(sig[1]),
            "k": len(sig[0]),
            "vertical_support": 0.0,
            "source_outcomes": [],
            "source_count": 0,
            "tirp_str": None,
        })
        rec["source_outcomes"].append(outcome)
        rec["source_count"] += 1
        if hasattr(row, "vertical_support") and pd.notna(row.vertical_support):
            rec["vertical_support"] = max(rec["vertical_support"], float(row.vertical_support))
        if rec["tirp_str"] is None and hasattr(row, "tirp_str"):
            rec["tirp_str"] = str(row.tirp_str)
    return merged


def pattern_signature_from_row(row):
    syms = row.symbols if not isinstance(row.symbols, str) else literal_eval(row.symbols)
    rels = row.relations if not isinstance(row.relations, str) else literal_eval(row.relations)
    return tuple(syms), tuple(rels)


def pattern_label_map(patterns_df):
    labels_by_key = {}
    if patterns_df is None or patterns_df.empty or "tirp_str" not in patterns_df.columns:
        return labels_by_key
    for row in patterns_df.itertuples(index=False):
        labels_by_key[pattern_signature_from_row(row)] = str(row.tirp_str)
    return labels_by_key


def build_combined_pattern_bank(patterns_by_outcome):
    merged = {}
    for outcome, patterns_df in patterns_by_outcome.items():
        if patterns_df is None or patterns_df.empty:
            continue
        for row in patterns_df.itertuples(index=False):
            sig = pattern_signature_from_row(row)
            rec = merged.setdefault(sig, {
                "symbols": list(sig[0]),
                "relations": list(sig[1]),
                "k": len(sig[0]),
                "vertical_support": 0.0,
                "source_outcomes": [],
                "source_count": 0,
                "tirp_str": None,
            })
            rec["source_outcomes"].append(outcome)
            rec["source_count"] += 1
            if hasattr(row, "vertical_support") and pd.notna(row.vertical_support):
                rec["vertical_support"] = max(rec["vertical_support"], float(row.vertical_support))
            if rec["tirp_str"] is None and hasattr(row, "tirp_str"):
                rec["tirp_str"] = str(row.tirp_str)

    rows = []
    for rec in merged.values():
        row = dict(rec)
        row["source_outcomes"] = tuple(sorted(set(row["source_outcomes"])))
        row["source_count"] = int(row["source_count"])
        rows.append(row)

    combined = pd.DataFrame(rows)
    if not combined.empty:
        combined = combined.sort_values(by=["k", "vertical_support"], ascending=[True, False]).reset_index(drop=True)
    return combined


def reconstruct_tirps(patterns_df):
    kl = KarmaLego(KL_EPSILON, KL_MAX_DISTANCE, KL_MIN_VER_SUPP, num_relations=KL_NUM_RELATIONS)
    tirps = []
    for row in patterns_df.itertuples(index=False):
        syms = row.symbols if not isinstance(row.symbols, str) else literal_eval(row.symbols)
        rels = row.relations if not isinstance(row.relations, str) else literal_eval(row.relations)
        tirps.append(TIRP(
            epsilon=kl.epsilon,
            max_distance=kl.max_distance,
            min_ver_supp=kl.min_ver_supp,
            symbols=list(syms),
            relations=list(rels),
            k=len(syms),
        ))
    return tirps


def apply_patterns_to_patient_set(obs_df, patient_ids, symbol_map, tirps, labels_by_key=None):
    known_df = filter_known_symbols(obs_df[obs_df["PatientId"].isin(patient_ids)].copy(), symbol_map)
    if known_df.empty or not tirps:
        return pd.DataFrame(index=pd.Index(patient_ids, name="PatientId"))

    pre = preprocess_dataframe(known_df, symbol_map)
    entities, aligned_ids = to_entity_list_for_patients(pre, patient_ids)

    kl = KarmaLego(KL_EPSILON, KL_MAX_DISTANCE, KL_MIN_VER_SUPP, num_relations=KL_NUM_RELATIONS)
    mode_values = kl.apply_patterns_to_entities_multi(
        entities,
        tirps,
        aligned_ids,
        modes=("tirp-count", "tpf-duration"),
        count_strategy="unique_last",
        n_jobs=APPLY_N_JOBS,
        show_progress=True,
    )
    count = mode_values.get("tirp-count", {})
    duration = mode_values.get("tpf-duration", {})

    labels_by_key = labels_by_key or {}
    rows = []
    for pid in aligned_ids:
        row = {"PatientId": pid}
        keys = set(count.get(pid, {}).keys()) | set(duration.get(pid, {}).keys())
        for key in keys:
            label = labels_by_key.get(key, str(key))
            row[f"tirp_count::{label}"] = count.get(pid, {}).get(key, 0.0)
            row[f"tpf_duration::{label}"] = duration.get(pid, {}).get(key, 0.0)
        rows.append(row)

    features = pd.DataFrame(rows).set_index("PatientId")
    features = features.reindex(patient_ids).fillna(0.0)
    return features


def shared_tirp_cache_fingerprint(patient_ids, tirps, chunk_size):
    payload = {
        "patient_ids": list(patient_ids),
        "chunk_size": int(chunk_size),
        "patterns": [
            {"symbols": list(t.symbols), "relations": list(t.relations)}
            for t in tirps
        ],
    }
    raw = json.dumps(payload, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def shared_tirp_manifest_path(chunk_dir):
    return chunk_dir / "manifest.json"


def load_shared_tirp_manifest(chunk_dir):
    path = shared_tirp_manifest_path(chunk_dir)
    if not path.exists():
        return None
    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as exc:
        print(f"Ignoring unreadable shared TIRP manifest: {exc}")
        return None


def write_shared_tirp_manifest(chunk_dir, fingerprint, patient_ids, tirps, chunk_size, chunk_files):
    manifest = {
        "fingerprint": fingerprint,
        "n_patients": len(patient_ids),
        "n_patterns": len(tirps),
        "chunk_size": int(chunk_size),
        "chunk_files": [p.name for p in chunk_files],
    }
    with shared_tirp_manifest_path(chunk_dir).open("w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)


def cached_shared_tirp_chunk_is_valid(chunk_path, chunk_ids, trust_manifest=False):
    if not chunk_path.exists() or chunk_path.stat().st_size == 0:
        return False
    if trust_manifest:
        return True
    try:
        cached = pd.read_pickle(chunk_path)
    except Exception as exc:
        print(f"Ignoring unreadable shared TIRP chunk {chunk_path.name}: {exc}")
        return False
    try:
        return list(cached.index) == list(chunk_ids)
    finally:
        del cached


def apply_shared_patterns_chunked(obs_df, patient_ids, tirps, labels_by_key, chunk_size, chunk_dir):
    chunk_dir.mkdir(parents=True, exist_ok=True)
    patient_ids = list(patient_ids)
    chunk_size = max(1, int(chunk_size))
    fingerprint = shared_tirp_cache_fingerprint(patient_ids, tirps, chunk_size)
    manifest = load_shared_tirp_manifest(chunk_dir)
    manifest_matches = manifest is not None and manifest.get("fingerprint") == fingerprint
    if manifest is not None and not manifest_matches:
        print("Shared TIRP cache manifest changed; recomputing chunks for the current pattern bank.")

    chunk_files = []
    reused = 0
    recomputed = 0
    for idx, start in enumerate(range(0, len(patient_ids), chunk_size)):
        chunk_ids = patient_ids[start:start + chunk_size]
        if not chunk_ids:
            continue
        chunk_path = chunk_dir / f"shared_tirp_chunk_{idx:04d}.pkl"
        can_reuse_chunk = manifest is None or manifest_matches
        if can_reuse_chunk and cached_shared_tirp_chunk_is_valid(chunk_path, chunk_ids, trust_manifest=manifest_matches):
            print(f"Reusing cached shared TIRP chunk {idx:04d} ({len(chunk_ids)} patients)")
            chunk_files.append(chunk_path)
            reused += 1
            continue

        print(f"Computing shared TIRP chunk {idx:04d} ({len(chunk_ids)} patients)")
        chunk_features = apply_patterns_to_patient_set(obs_df, chunk_ids, shared_symbol_map, tirps, labels_by_key)
        chunk_features.to_pickle(chunk_path)
        chunk_files.append(chunk_path)
        recomputed += 1
        del chunk_features
        gc.collect()

    write_shared_tirp_manifest(chunk_dir, fingerprint, patient_ids, tirps, chunk_size, chunk_files)
    print(f"Shared TIRP chunk cache: reused {reused}, computed {recomputed}")
    return chunk_files


def as_sparse_float32_frame(df):
    if df.empty:
        return df.astype(np.float32)
    if all(isinstance(dtype, pd.SparseDtype) for dtype in df.dtypes):
        return df.astype(pd.SparseDtype("float32", 0.0))
    out = df.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    return out.astype(pd.SparseDtype("float32", 0.0))


def nonzero_columns_blockwise(df, block_size=2048):
    keep = []
    for start in range(0, df.shape[1], block_size):
        block = df.iloc[:, start:start + block_size]
        mask = block.ne(0).any(axis=0)
        keep.extend(mask.index[mask].tolist())
        del block, mask
    return pd.Index(keep)


def sparse_tirp_cache_paths(chunk_path):
    cache_dir = chunk_path.parent / "sparse_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    stem = chunk_path.stem
    return cache_dir / f"{stem}.npz", cache_dir / f"{stem}.meta.pkl"


def load_or_create_sparse_tirp_chunk(chunk_path):
    matrix_path, meta_path = sparse_tirp_cache_paths(chunk_path)
    if matrix_path.exists() and meta_path.exists():
        meta = pd.read_pickle(meta_path)
        return meta["patient_ids"], meta["columns"], sp.load_npz(matrix_path).tocsr().astype(np.float32)

    print(f"Converting {chunk_path.name} to compact sparse cache")
    dense = pd.read_pickle(chunk_path)
    patient_ids = list(dense.index)
    matrices = []
    columns = []
    block_size = 2048
    for start in range(0, dense.shape[1], block_size):
        block = dense.iloc[:, start:start + block_size]
        keep_cols = block.columns[block.ne(0).any(axis=0)]
        if len(keep_cols):
            arr = block.loc[:, keep_cols].to_numpy(dtype=np.float32, copy=True)
            matrices.append(sp.csr_matrix(arr))
            columns.extend(keep_cols.tolist())
            del arr
        del block, keep_cols
    if matrices:
        matrix = sp.hstack(matrices, format="csr", dtype=np.float32)
    else:
        matrix = sp.csr_matrix((len(patient_ids), 0), dtype=np.float32)
    sp.save_npz(matrix_path, matrix)
    pd.to_pickle({"patient_ids": patient_ids, "columns": columns}, meta_path)
    del dense, matrices
    gc.collect()
    return patient_ids, columns, matrix


def load_shared_tirp_subset(patient_ids, chunk_dir):
    X, columns = load_shared_tirp_matrix(patient_ids, chunk_dir)
    return pd.DataFrame.sparse.from_spmatrix(
        X,
        index=pd.Index(patient_ids, name="PatientId"),
        columns=columns,
    )


def load_shared_tirp_matrix(patient_ids, chunk_dir):
    patient_ids = list(patient_ids)
    if not patient_ids:
        return sp.csr_matrix((0, 0), dtype=np.float32), []

    wanted_pos = {pid: i for i, pid in enumerate(patient_ids)}
    global_col_pos = {}
    global_cols = []
    row_parts = []
    col_parts = []
    data_parts = []

    for chunk_path in sorted(chunk_dir.glob("shared_tirp_chunk_*.pkl")):
        chunk_ids, chunk_cols, chunk_matrix = load_or_create_sparse_tirp_chunk(chunk_path)
        selected = [(r, wanted_pos[pid]) for r, pid in enumerate(chunk_ids) if pid in wanted_pos]
        if not selected or chunk_matrix.shape[1] == 0:
            del chunk_matrix
            continue

        local_rows, target_rows = zip(*selected)
        sub = chunk_matrix[list(local_rows), :].tocoo()
        if sub.nnz:
            mapped_cols = np.empty(sub.nnz, dtype=np.int64)
            for local_col in np.unique(sub.col):
                feature = chunk_cols[int(local_col)]
                pos = global_col_pos.get(feature)
                if pos is None:
                    pos = len(global_cols)
                    global_col_pos[feature] = pos
                    global_cols.append(feature)
                mapped_cols[sub.col == local_col] = pos
            target_rows = np.asarray(target_rows, dtype=np.int64)
            row_parts.append(target_rows[sub.row])
            col_parts.append(mapped_cols)
            data_parts.append(sub.data.astype(np.float32, copy=False))
        del chunk_matrix, sub
        gc.collect()

    if not global_cols:
        return sp.csr_matrix((len(patient_ids), 0), dtype=np.float32), []
    rows = np.concatenate(row_parts) if row_parts else np.array([], dtype=np.int64)
    cols = np.concatenate(col_parts) if col_parts else np.array([], dtype=np.int64)
    data = np.concatenate(data_parts) if data_parts else np.array([], dtype=np.float32)
    matrix = sp.csr_matrix((data, (rows, cols)), shape=(len(patient_ids), len(global_cols)), dtype=np.float32)
    matrix.sum_duplicates()
    return matrix, global_cols


def load_shared_tirp_train_test(train_ids, test_ids, chunk_dir):
    if not USE_COMPACT_SPARSE_TIRP_CACHE:
        return load_shared_tirp_subset(train_ids, chunk_dir), load_shared_tirp_subset(test_ids, chunk_dir)

    all_ids = list(train_ids) + list(test_ids)
    all_matrix, all_columns = load_shared_tirp_matrix(all_ids, chunk_dir)
    train_n = len(train_ids)
    X_train = all_matrix[:train_n, :]
    X_test = all_matrix[train_n:, :]
    X_train_df = pd.DataFrame.sparse.from_spmatrix(
        X_train,
        index=pd.Index(train_ids, name="PatientId"),
        columns=all_columns,
    )
    X_test_df = pd.DataFrame.sparse.from_spmatrix(
        X_test,
        index=pd.Index(test_ids, name="PatientId"),
        columns=all_columns,
    )
    print("Loaded compact sparse TIRP matrix:", all_matrix.shape, "non-zero values:", int(all_matrix.nnz))
    return X_train_df, X_test_df


def feature_columns_for_patterns(patterns_df):
    labels_by_key = pattern_label_map(patterns_df)
    labels = list(labels_by_key.values())
    return sorted({
        col
        for label in labels
        for col in (f"tirp_count::{label}", f"tpf_duration::{label}")
    })


def filtered_tirp_cache_dir(output_dir, min_vertical_support):
    suffix = f"mvs_{min_vertical_support:.2f}".replace(".", "_")
    return output_dir / f"filtered_tirp_cache_{suffix}"


def filtered_tirp_cache_paths(cache_dir):
    return {
        "matrix": cache_dir / "matrix.npz",
        "patient_ids": cache_dir / "patient_ids.pkl",
        "columns": cache_dir / "columns.pkl",
        "manifest": cache_dir / "manifest.json",
    }


def sparse_value_dtype():
    return np.dtype(FILTERED_TIRP_VALUE_DTYPE)


def build_or_load_filtered_tirp_cache(chunk_dir, keep_columns, patient_ids, cache_dir, min_vertical_support):
    cache_dir.mkdir(parents=True, exist_ok=True)
    paths = filtered_tirp_cache_paths(cache_dir)
    keep_columns = list(keep_columns)
    wanted_patient_ids = list(patient_ids)
    expected = {
        "min_vertical_support": float(min_vertical_support),
        "n_keep_columns": len(keep_columns),
        "n_patients": len(wanted_patient_ids),
        "value_dtype": str(sparse_value_dtype()),
    }

    if all(paths[k].exists() for k in ("matrix", "patient_ids", "columns", "manifest")):
        try:
            with paths["manifest"].open("r", encoding="utf-8") as f:
                manifest = json.load(f)
            if all(manifest.get(k) == v for k, v in expected.items()):
                matrix = sp.load_npz(paths["matrix"]).tocsr().astype(sparse_value_dtype())
                cached_patient_ids = pd.read_pickle(paths["patient_ids"])
                cached_columns = pd.read_pickle(paths["columns"])
                if list(cached_patient_ids) == wanted_patient_ids and list(cached_columns) == keep_columns:
                    print(f"[cache] Loaded filtered TIRP cache from {paths['matrix']}: shape={matrix.shape}, nnz={int(matrix.nnz)}")
                    return matrix, cached_patient_ids, cached_columns
        except Exception as exc:
            print(f"Ignoring filtered TIRP cache and rebuilding: {exc}")

    value_dtype = sparse_value_dtype()
    print(f"[cache] Building filtered TIRP cache from dense chunks in {chunk_dir}")
    print(f"[cache] Destination: {cache_dir}")
    print(f"[cache] Filtered columns={len(keep_columns)}, patients={len(wanted_patient_ids)}, dtype={value_dtype}")
    patient_pos = {pid: i for i, pid in enumerate(wanted_patient_ids)}
    col_pos = {col: i for i, col in enumerate(keep_columns)}
    keep_set = set(keep_columns)
    row_parts = []
    col_parts = []
    data_parts = []

    for chunk_path in sorted(chunk_dir.glob("shared_tirp_chunk_*.pkl")):
        chunk_df = pd.read_pickle(chunk_path)
        cols = [c for c in chunk_df.columns if c in keep_set]
        rows_mask = chunk_df.index.isin(patient_pos)
        if not cols or not rows_mask.any():
            print(f"[cache] {chunk_path.name}: kept_columns={len(cols)}, nnz=0")
            del chunk_df, rows_mask
            gc.collect()
            continue

        sub = chunk_df.loc[rows_mask, cols]
        del chunk_df, rows_mask
        gc.collect()

        row_indexer = np.asarray([patient_pos[pid] for pid in sub.index], dtype=np.int64)
        values = sub.to_numpy(dtype=value_dtype, copy=True)
        del sub
        gc.collect()

        coo = sp.coo_matrix(values)
        del values
        if coo.nnz:
            row_parts.append(row_indexer[coo.row])
            col_parts.append(np.asarray([col_pos[cols[j]] for j in coo.col], dtype=np.int64))
            data_parts.append(coo.data.astype(value_dtype, copy=False))
        print(f"[cache] {chunk_path.name}: kept_columns={len(cols)}, nnz={int(coo.nnz)}")
        del row_indexer, coo
        gc.collect()

    if row_parts:
        rows = np.concatenate(row_parts)
        cols = np.concatenate(col_parts)
        data = np.concatenate(data_parts)
    else:
        rows = np.array([], dtype=np.int64)
        cols = np.array([], dtype=np.int64)
        data = np.array([], dtype=value_dtype)
    matrix = sp.csr_matrix((data, (rows, cols)), shape=(len(wanted_patient_ids), len(keep_columns)), dtype=value_dtype)
    matrix.sum_duplicates()

    sp.save_npz(paths["matrix"], matrix)
    pd.to_pickle(wanted_patient_ids, paths["patient_ids"])
    pd.to_pickle(keep_columns, paths["columns"])
    manifest = {**expected, "nnz": int(matrix.nnz)}
    with paths["manifest"].open("w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    print(f"[cache] Wrote filtered TIRP cache to {paths['matrix']}: shape={matrix.shape}, nnz={int(matrix.nnz)}")
    return matrix, wanted_patient_ids, keep_columns


def load_shared_tirp_train_test(train_ids, test_ids, chunk_dir, keep_columns=None):
    all_ids = list(train_ids) + list(test_ids)
    if keep_columns is None:
        raise ValueError("keep_columns is required for the filtered single-cache loader")
    cache_dir = filtered_tirp_cache_dir(OUTPUT_DIR, MIN_SHARED_PATTERN_VERTICAL_SUPPORT)
    matrix, cached_patient_ids, columns = build_or_load_filtered_tirp_cache(
        chunk_dir,
        keep_columns,
        all_ids,
        cache_dir,
        MIN_SHARED_PATTERN_VERTICAL_SUPPORT,
    )
    train_n = len(train_ids)
    X_train = matrix[:train_n, :]
    X_test = matrix[train_n:, :]
    X_train_df = pd.DataFrame.sparse.from_spmatrix(
        X_train,
        index=pd.Index(train_ids, name="PatientId"),
        columns=columns,
    )
    X_test_df = pd.DataFrame.sparse.from_spmatrix(
        X_test,
        index=pd.Index(test_ids, name="PatientId"),
        columns=columns,
    )
    print(f"[features] Split filtered TIRP matrix into train={X_train_df.shape}, test={X_test_df.shape}")
    return X_train_df, X_test_df


def add_train_fitted_minmax(train_df, test_df, prefix="tirp_count::"):
    count_cols = [c for c in train_df.columns if c.startswith(prefix)]
    if not count_cols:
        return train_df, test_df
    train_out = train_df.copy()
    test_out = test_df.copy()
    mins = train_df[count_cols].min(axis=0)
    maxs = train_df[count_cols].max(axis=0)
    denom = (maxs - mins).replace(0, np.nan)
    train_norm = (train_df[count_cols] - mins) / denom
    test_norm = (test_df[count_cols] - mins) / denom
    train_norm = train_norm.fillna(0.0).clip(lower=0.0, upper=1.0)
    test_norm = test_norm.fillna(0.0).clip(lower=0.0, upper=1.0)
    train_norm.columns = ["count_train_minmax::" + c[len(prefix):] for c in count_cols]
    test_norm.columns = train_norm.columns
    return pd.concat([train_out, train_norm], axis=1), pd.concat([test_out, test_norm], axis=1)


## 5. Optional QA and Context Feature Helpers


In [7]:
def load_context_features(patient_ids_train, patient_ids_test):
    if not USE_CONTEXT_DATA or not CONTEXT_DATA_PATH.exists():
        return (
            pd.DataFrame(index=pd.Index(patient_ids_train, name="PatientId")),
            pd.DataFrame(index=pd.Index(patient_ids_test, name="PatientId")),
        )
    ctx = pd.read_csv(CONTEXT_DATA_PATH)
    if "PatientId" not in ctx.columns:
        raise ValueError("context_data.csv must contain PatientId")
    ctx = ctx.drop_duplicates("PatientId").set_index("PatientId")
    ctx = pd.get_dummies(ctx, dummy_na=True)
    train = ctx.reindex(patient_ids_train)
    test = ctx.reindex(patient_ids_test)
    return train, test


def load_qa_features(patient_ids_train, patient_ids_test):
    if not USE_QA_DATA or not QA_DATA_PATH.exists():
        return (
            pd.DataFrame(index=pd.Index(patient_ids_train, name="PatientId")),
            pd.DataFrame(index=pd.Index(patient_ids_test, name="PatientId")),
        )
    qa = pd.read_csv(QA_DATA_PATH)
    if "PatientId" not in qa.columns:
        raise ValueError("qa_data.csv must contain PatientId")

    if "HospitalizationDay" in qa.columns:
        qa = qa[pd.to_numeric(qa["HospitalizationDay"], errors="coerce") <= OBS_DAYS].copy()

    # Preferred shape: PatientId, PatternName, Value. Fall back to numeric aggregates.
    if {"PatternName", "Value"}.issubset(qa.columns):
        qa["Value"] = pd.to_numeric(qa["Value"], errors="coerce")
        features = qa.pivot_table(index="PatientId", columns="PatternName", values="Value", aggfunc="mean")
        features.columns = [f"qa_mean::{c}" for c in features.columns]
    else:
        numeric_cols = [c for c in qa.select_dtypes(include=[np.number]).columns if c != "PatientId"]
        features = qa.groupby("PatientId")[numeric_cols].mean()
        features.columns = [f"qa_mean::{c}" for c in features.columns]

    train = features.reindex(patient_ids_train)
    test = features.reindex(patient_ids_test)
    return train, test


def is_sparse_frame(df):
    return not df.empty and all(isinstance(dtype, pd.SparseDtype) for dtype in df.dtypes)


def ensure_sparse_float32_frame(df):
    if df.empty:
        return df.astype(np.float32)
    if is_sparse_frame(df):
        return df.astype(pd.SparseDtype("float32", 0.0))
    return df.apply(pd.to_numeric, errors="coerce").fillna(0.0).astype(pd.SparseDtype("float32", 0.0))


def combine_feature_blocks(train_blocks, test_blocks):
    train_blocks = [ensure_sparse_float32_frame(block) for block in train_blocks]
    test_blocks = [ensure_sparse_float32_frame(block) for block in test_blocks]
    X_train = pd.concat(train_blocks, axis=1, sort=False)
    X_test = pd.concat(test_blocks, axis=1, sort=False)
    X_train, X_test = X_train.align(X_test, join="outer", axis=1, fill_value=0.0)
    X_train = ensure_sparse_float32_frame(X_train.fillna(0.0))
    X_test = ensure_sparse_float32_frame(X_test.fillna(0.0))
    return X_train, X_test


def nonzero_feature_columns(df):
    if df.empty:
        return pd.Index([])
    if is_sparse_frame(df):
        coo = df.sparse.to_coo()
        return df.columns[np.asarray(coo.getnnz(axis=0)).ravel() > 0]
    return nonzero_columns_blockwise(df)


## 6. Modeling Helpers


In [8]:

def sanitize_feature_columns(X_train, X_test):
    original_cols = list(X_train.columns)
    safe_cols = [f"f_{i:06d}" for i in range(len(original_cols))]
    feature_map = pd.DataFrame({"safe_feature": safe_cols, "original_feature": original_cols})
    train_safe = X_train.copy()
    test_safe = X_test.copy()
    train_safe.columns = safe_cols
    test_safe.columns = safe_cols
    return train_safe, test_safe, feature_map


def make_xgb_model(y_train, **overrides):
    pos = int(np.sum(y_train == 1))
    neg = int(np.sum(y_train == 0))
    scale_pos_weight = (neg / pos) if pos else 1.0
    params = dict(
        n_estimators=250,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=1,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
    params.update(overrides)
    return XGBClassifier(**params)


def make_lr_model():
    return Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            solver="saga",
            penalty="l2",
            C=1.0,
            max_iter=500,
            tol=1e-3,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1,
        )),
    ])


def make_model(y_train):
    if HAS_XGBOOST:
        return make_xgb_model(y_train)
    return make_lr_model()


def to_model_matrix(X):
    if isinstance(X, pd.DataFrame) and is_sparse_frame(X):
        return X.sparse.to_coo().tocsr().astype(np.float32)
    if sp.issparse(X):
        return X.tocsr().astype(np.float32)
    return sp.csr_matrix(np.asarray(X, dtype=np.float32))


def fit_estimator(model, X, y):
    model.fit(to_model_matrix(X), y)
    return model


def predict_scores(model, X_test):
    X_mat = to_model_matrix(X_test)
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_mat)[:, 1]
    return model.decision_function(X_mat)


def best_f1(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = (2 * precision * recall) / np.maximum(precision + recall, 1e-12)
    idx = int(np.nanargmax(f1))
    threshold = thresholds[idx - 1] if idx > 0 and idx - 1 < len(thresholds) else 0.5
    return float(f1[idx]), float(threshold)


def evaluate_binary(y_true, scores):
    out = {}
    out["n"] = int(len(y_true))
    out["positives"] = int(np.sum(y_true == 1))
    out["controls"] = int(np.sum(y_true == 0))
    out["pr_auc"] = average_precision_score(y_true, scores) if len(np.unique(y_true)) > 1 else np.nan
    out["roc_auc"] = roc_auc_score(y_true, scores) if len(np.unique(y_true)) > 1 else np.nan
    out["f1_at_0_5"] = f1_score(y_true, scores >= 0.5) if len(np.unique(y_true)) > 1 else np.nan
    out["best_f1"], out["best_f1_threshold"] = best_f1(y_true, scores) if len(np.unique(y_true)) > 1 else (np.nan, np.nan)
    return out


def gain_importance_series(model, feature_names):
    booster = model.get_booster()
    score = booster.get_score(importance_type="gain")
    imp = pd.Series(0.0, index=list(feature_names))
    for feature, value in score.items():
        if feature in imp.index:
            imp.loc[feature] = float(value)
        elif isinstance(feature, str) and feature.startswith("f") and feature[1:].isdigit():
            pos = int(feature[1:])
            if pos < len(imp):
                imp.iloc[pos] = float(value)
    return imp.sort_values(ascending=False)


def prune_features_by_gain(X_train, y_train):
    if not HAS_XGBOOST or not ENABLE_FEATURE_PRUNING or X_train.shape[1] == 0:
        return list(X_train.columns), pd.DataFrame()

    baseline = make_xgb_model(y_train, n_estimators=200)
    fit_estimator(baseline, X_train, y_train)
    gains = gain_importance_series(baseline, X_train.columns)
    positive_gains = gains[gains > 0]

    if positive_gains.empty:
        selected = list(X_train.columns)
    else:
        cumulative = positive_gains.cumsum() / positive_gains.sum()
        keep_n = int(np.searchsorted(cumulative.to_numpy(), GAIN_KEEP_FRACTION, side="left") + 1)
        keep_n = max(MIN_FEATURES_AFTER_PRUNING, keep_n)
        keep_n = min(keep_n, len(positive_gains))
        selected = list(positive_gains.index[:keep_n])

    report = pd.DataFrame({
        "feature": gains.index,
        "gain": gains.values,
    })
    report["selected"] = report["feature"].isin(selected)
    return selected, report


def can_make_validation_split(y_train):
    counts = pd.Series(y_train).value_counts()
    return len(counts) == 2 and counts.min() >= 4 and len(y_train) >= 30


def tune_xgb_with_optuna(X_train, y_train):
    if not (HAS_XGBOOST and HAS_OPTUNA and ENABLE_OPTUNA):
        return {}, pd.DataFrame()
    if not can_make_validation_split(y_train):
        print("Skipping Optuna: not enough samples/positives for a validation split.")
        return {}, pd.DataFrame()

    X_fit, X_val, y_fit, y_val = train_test_split(
        X_train,
        y_train,
        test_size=0.25,
        random_state=RANDOM_STATE,
        stratify=y_train,
    )

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "max_depth": trial.suggest_int("max_depth", 2, 5),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "subsample": trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
        }
        model = make_xgb_model(y_fit, **params)
        fit_estimator(model, X_fit, y_fit)
        scores = predict_scores(model, X_val)
        return average_precision_score(y_val, scores)

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=OPTUNA_TRIALS, timeout=OPTUNA_TIMEOUT_SECONDS, show_progress_bar=False)

    trials = study.trials_dataframe()
    return study.best_params, trials


def fit_model_with_optional_pruning_and_tuning(X_train, y_train):
    y_arr = np.asarray(y_train)

    selected_features, pruning_report = prune_features_by_gain(X_train, y_arr)
    X_selected = X_train[selected_features]

    best_params, optuna_trials = tune_xgb_with_optuna(X_selected, y_arr)

    if HAS_XGBOOST:
        model = make_xgb_model(y_arr, **best_params)
    else:
        model = make_lr_model()
    fit_estimator(model, X_selected, y_arr)
    return model, selected_features, pruning_report, optuna_trials, best_params


def top_model_features(model, feature_names, feature_map=None, top_n=30):
    if HAS_XGBOOST and hasattr(model, "get_booster"):
        imp = gain_importance_series(model, feature_names).head(top_n).rename("importance").reset_index(names="safe_feature")
        if feature_map is not None:
            imp = imp.merge(feature_map, on="safe_feature", how="left")
        return imp
    if isinstance(model, Pipeline):
        clf = model.named_steps["model"]
        coef = pd.Series(np.abs(clf.coef_[0]), index=feature_names).sort_values(ascending=False)
        out = coef.head(top_n).rename("abs_coef").reset_index(names="safe_feature")
        if feature_map is not None:
            out = out.merge(feature_map, on="safe_feature", how="left")
        return out
    return pd.DataFrame()



def original_feature_names(safe_features, feature_map):
    mapper = dict(zip(feature_map["safe_feature"], feature_map["original_feature"]))
    return [mapper.get(f, f) for f in safe_features]


def compute_shap_outputs(model, X_selected, feature_map, task_dir, outcome):
    """
    Save SHAP feature importance and a beeswarm plot for the final selected model.

    For binary models, positive SHAP values push toward the outcome class and
    negative values push toward the no-outcome class. The saved CSV includes both
    mean absolute SHAP and signed mean SHAP.
    """
    if not (ENABLE_SHAP and HAS_SHAP and HAS_XGBOOST and hasattr(model, "get_booster")):
        return pd.DataFrame()
    if X_selected.empty:
        return pd.DataFrame()

    sample = X_selected
    if len(sample) > SHAP_MAX_SAMPLES:
        sample = sample.sample(SHAP_MAX_SAMPLES, random_state=RANDOM_STATE)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(to_model_matrix(sample))
    if isinstance(shap_values, list):
        shap_values = shap_values[-1]
    shap_values = np.asarray(shap_values)

    original_names = original_feature_names(list(sample.columns), feature_map)
    shap_df = pd.DataFrame(shap_values, index=sample.index, columns=original_names)
    display_df = sample.sparse.to_dense() if isinstance(sample, pd.DataFrame) and is_sparse_frame(sample) else sample.copy()
    display_df.columns = original_names

    importance = pd.DataFrame({
        "outcome": outcome,
        "feature": original_names,
        "mean_abs_shap": np.abs(shap_values).mean(axis=0),
        "mean_shap": shap_values.mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False)
    importance["direction"] = np.where(
        importance["mean_shap"] >= 0,
        "pushes_outcome",
        "pushes_no_outcome",
    )
    importance.to_csv(task_dir / "shap_feature_importance.csv", index=False)

    # Beeswarm on the top selected features only, to keep plots readable.
    top_features = importance.head(SHAP_TOP_N)["feature"].tolist()
    top_idx = [original_names.index(f) for f in top_features]
    plt.figure(figsize=(10, max(5, min(14, len(top_features) * 0.35))))
    shap.summary_plot(
        shap_values[:, top_idx],
        display_df[top_features],
        show=False,
        max_display=SHAP_TOP_N,
    )
    plt.title(f"SHAP beeswarm: {outcome}")
    plt.tight_layout()
    plt.savefig(task_dir / "shap_beeswarm.png", dpi=200, bbox_inches="tight")
    plt.close()

    return importance


def make_multilabel_linear_model():
    return OneVsRestClassifier(make_lr_model(), n_jobs=1)


def predict_multilabel_scores(model, X_test):
    X_mat = to_model_matrix(X_test)
    if hasattr(model, "predict_proba"):
        scores = model.predict_proba(X_mat)
    else:
        scores = model.decision_function(X_mat)
    return np.asarray(scores)


def evaluate_multilabel(Y_true, scores, outcomes):
    Y_true = pd.DataFrame(Y_true, columns=outcomes).astype(int)
    scores = np.asarray(scores)
    rows = []
    for j, outcome in enumerate(outcomes):
        y = Y_true[outcome].to_numpy()
        s = scores[:, j]
        row = evaluate_binary(y, s)
        row["outcome"] = outcome
        rows.append(row)

    per_outcome = pd.DataFrame(rows)
    pred_05 = (scores >= 0.5).astype(int)
    summary = {
        "model": "one_vs_rest_logistic_regression",
        "n_test": int(Y_true.shape[0]),
        "n_labels": int(Y_true.shape[1]),
        "micro_pr_auc": average_precision_score(Y_true.to_numpy().ravel(), scores.ravel()),
        "macro_pr_auc": float(per_outcome["pr_auc"].mean()),
        "micro_f1_at_0_5": f1_score(Y_true.to_numpy(), pred_05, average="micro", zero_division=0),
        "macro_f1_at_0_5": f1_score(Y_true.to_numpy(), pred_05, average="macro", zero_division=0),
    }
    try:
        summary["micro_roc_auc"] = roc_auc_score(Y_true.to_numpy(), scores, average="micro")
        summary["macro_roc_auc"] = roc_auc_score(Y_true.to_numpy(), scores, average="macro")
    except ValueError:
        summary["micro_roc_auc"] = np.nan
        summary["macro_roc_auc"] = np.nan
    return summary, per_outcome


def top_multilabel_linear_features(model, feature_names, outcomes, feature_map=None, top_n=30):
    rows = []
    mapper = dict(zip(feature_map["safe_feature"], feature_map["original_feature"])) if feature_map is not None else {}
    for outcome, estimator in zip(outcomes, model.estimators_):
        clf = estimator.named_steps["model"] if isinstance(estimator, Pipeline) else estimator
        if not hasattr(clf, "coef_"):
            continue
        coef = pd.Series(clf.coef_[0], index=feature_names)
        top = coef.abs().sort_values(ascending=False).head(top_n).index
        for safe_feature in top:
            rows.append({
                "outcome": outcome,
                "safe_feature": safe_feature,
                "original_feature": mapper.get(safe_feature, safe_feature),
                "coef": float(coef.loc[safe_feature]),
                "abs_coef": float(abs(coef.loc[safe_feature])),
            })
    return pd.DataFrame(rows)


def plot_outcome_shap_grid(shap_importance_by_outcome, output_dir, top_n_per_outcome=10):
    """
    Build a compact outcome x feature heatmap using mean_abs_shap.
    """
    frames = [df for df in shap_importance_by_outcome.values() if isinstance(df, pd.DataFrame) and not df.empty]
    if not frames:
        print("No SHAP importance tables available for grid plot.")
        return pd.DataFrame()

    all_imp = pd.concat(frames, ignore_index=True)
    top_features = (
        all_imp.sort_values("mean_abs_shap", ascending=False)
        .groupby("outcome")
        .head(top_n_per_outcome)["feature"]
        .drop_duplicates()
        .tolist()
    )
    grid = (
        all_imp[all_imp["feature"].isin(top_features)]
        .pivot_table(index="outcome", columns="feature", values="mean_abs_shap", aggfunc="max", fill_value=0.0)
    )
    grid = grid.loc[:, grid.max(axis=0).sort_values(ascending=False).index]
    grid.to_csv(output_dir / "shap_outcome_feature_grid.csv")

    plt.figure(figsize=(max(10, 0.35 * grid.shape[1]), max(4, 0.6 * grid.shape[0])))
    plt.imshow(grid.to_numpy(), aspect="auto", cmap="viridis")
    plt.colorbar(label="mean |SHAP|")
    plt.yticks(range(grid.shape[0]), grid.index)
    plt.xticks(range(grid.shape[1]), grid.columns, rotation=90, fontsize=8)
    plt.title("Outcome x feature SHAP importance grid")
    plt.tight_layout()
    plt.savefig(output_dir / "shap_outcome_feature_grid.png", dpi=200, bbox_inches="tight")
    plt.close()
    return grid


## 7. Discovery and Shared Feature Build


In [9]:
def run_multilabel_pipeline(outcomes, shared_tirp_chunk_dir, shared_pattern_count, keep_tirp_feature_columns=None):
    print("\n=== Multi-label model ===")
    task_dir = OUTPUT_DIR / "multilabel_model"
    task_dir.mkdir(parents=True, exist_ok=True)
    print(f"[setup] Output directory: {task_dir}")

    print("[split] Creating multi-label train/test split")
    train_ids, test_ids, Y_all = split_for_multilabel(labels, obs_counts, outcomes)
    Y_train = Y_all.reindex(train_ids).astype(int)
    Y_test = Y_all.reindex(test_ids).astype(int)

    print(f"[split] Train patients={len(train_ids)}, test patients={len(test_ids)}, labels={len(outcomes)}")
    print("[split] Train positives by outcome:")
    display(Y_train.sum().rename("positives").to_frame())
    print("[split] Test positives by outcome:")
    display(Y_test.sum().rename("positives").to_frame())

    print(f"[features] Loading TIRP features from {shared_tirp_chunk_dir}")
    X_train_tirp, X_test_tirp = load_shared_tirp_train_test(train_ids, test_ids, shared_tirp_chunk_dir, keep_tirp_feature_columns)

    if USE_COUNT_MINMAX_FROM_TRAIN:
        X_train_tirp, X_test_tirp = add_train_fitted_minmax(X_train_tirp, X_test_tirp)

    print("[features] Loading optional QA/context features")
    X_train_qa, X_test_qa = load_qa_features(train_ids, test_ids)
    X_train_ctx, X_test_ctx = load_context_features(train_ids, test_ids)

    print("[features] Combining feature blocks")
    X_train, X_test = combine_feature_blocks(
        [X_train_tirp, X_train_qa, X_train_ctx],
        [X_test_tirp, X_test_qa, X_test_ctx],
    )

    nonzero_cols = nonzero_feature_columns(X_train)
    X_train = ensure_sparse_float32_frame(X_train.loc[:, nonzero_cols])
    X_test = ensure_sparse_float32_frame(X_test.reindex(columns=nonzero_cols, fill_value=0.0))
    print("[features] Final feature matrix:", X_train.shape, "non-zero values:", int(X_train.sparse.to_coo().nnz) if is_sparse_frame(X_train) else int((X_train != 0).sum().sum()))

    print("[features] Sanitizing feature names for model compatibility")
    X_train_model, X_test_model, feature_map = sanitize_feature_columns(X_train, X_test)
    selected_features = list(X_train_model.columns)
    feature_map.to_csv(task_dir / "feature_name_map.csv", index=False)

    model = make_multilabel_linear_model()
    print(f"[train] Training {MODEL_FAMILY} one-vs-rest model on {X_train_model.shape[0]} patients x {X_train_model.shape[1]} features")
    model.fit(to_model_matrix(X_train_model), Y_train.to_numpy())

    print(f"[evaluate] Scoring {X_test_model.shape[0]} test patients")
    scores = predict_multilabel_scores(model, X_test_model)
    print("[evaluate] Computing multi-label and per-outcome metrics")
    summary_metrics, per_outcome_metrics = evaluate_multilabel(Y_test, scores, outcomes)
    summary_metrics.update({
        "n_train": len(train_ids),
        "n_features_raw": X_train.shape[1],
        "n_patterns_shared": int(shared_pattern_count),
        "label_window": f"> {OBS_HOURS}h" if LABEL_AFTER_OBS_WINDOW else f"0..{OBS_HOURS}h",
        "label_end_hours": LABEL_END_HOURS,
    })

    print(f"[write] Writing metrics, predictions, matrices, and model to {task_dir}")
    pd.Series(summary_metrics).to_json(task_dir / "metrics_summary.json", indent=2)
    per_outcome_metrics.to_csv(task_dir / "metrics_by_outcome.csv", index=False)
    pd.DataFrame(scores, index=Y_test.index, columns=outcomes).to_csv(task_dir / "test_scores.csv")
    (scores >= 0.5).astype(int)
    pd.DataFrame((scores >= 0.5).astype(int), index=Y_test.index, columns=outcomes).to_csv(task_dir / "test_predictions_at_0_5.csv")
    Y_train.to_csv(task_dir / "Y_train.csv")
    Y_test.to_csv(task_dir / "Y_test.csv")
    X_train.to_pickle(task_dir / "X_train.pkl")
    X_test.to_pickle(task_dir / "X_test.pkl")
    X_train_model.to_pickle(task_dir / "X_train_model.pkl")
    X_test_model.to_pickle(task_dir / "X_test_model.pkl")
    joblib.dump(model, task_dir / "model.joblib")

    top_features = top_multilabel_linear_features(model, selected_features, outcomes, feature_map=feature_map, top_n=SHAP_TOP_N)
    if not top_features.empty:
        top_features.to_csv(task_dir / "top_features_by_outcome.csv", index=False)

    print("[done] Summary metrics:", summary_metrics)
    display(per_outcome_metrics)
    return summary_metrics, per_outcome_metrics, top_features


## 8. Outcome Modeling


In [10]:
print("[patterns] Loading cached per-outcome pattern files")
combined_patterns_merged = {}

for outcome in OUTCOMES_TO_RUN:
    task_dir = OUTPUT_DIR / outcome.replace("/", "_")
    patterns_path = task_dir / "discovered_patterns.csv"

    if patterns_path.exists() and patterns_path.stat().st_size > 0:
        print(f"[patterns] Loading cached patterns for {outcome}")
        patterns_df = load_patterns_dataframe(patterns_path)
        combined_patterns_merged = update_combined_pattern_bank(combined_patterns_merged, outcome, patterns_df)
        del patterns_df
        gc.collect()
        continue

    if "observation_temporal" not in globals():
        raise NameError(
            "observation_temporal is not defined and cached patterns are missing. "
            "Rerun the data-preparation cell before recomputing patterns."
        )

    print(f"[patterns] Cache missing for {outcome}; recomputing patterns")
    train_ids, _, y_all = split_for_outcome(labels, obs_counts, outcome)
    y_train = y_all.reindex(train_ids).astype(int)
    train_lengths = admission_lengths.reindex(train_ids)
    train_obs = observation_temporal[observation_temporal["PatientId"].isin(train_ids)].copy()
    patterns_df = discover_or_load_patterns_for_outcome(
        train_obs,
        train_ids,
        y_train,
        train_lengths,
        task_dir,
        outcome,
    )
    combined_patterns_merged = update_combined_pattern_bank(combined_patterns_merged, outcome, patterns_df)
    del train_obs, train_ids, y_all, y_train, train_lengths, patterns_df
    gc.collect()

combined_patterns_df = pd.DataFrame([
    {
        **rec,
        "source_outcomes": tuple(sorted(set(rec["source_outcomes"]))),
        "source_count": int(rec["source_count"]),
    }
    for rec in combined_patterns_merged.values()
])
if not combined_patterns_df.empty:
    combined_patterns_df = combined_patterns_df.sort_values(by=["k", "vertical_support"], ascending=[True, False]).reset_index(drop=True)
combined_patterns_df.to_csv(OUTPUT_DIR / "combined_deduped_patterns.csv", index=False)
print(f"[patterns] Wrote combined pattern bank to {OUTPUT_DIR / 'combined_deduped_patterns.csv'}")

all_shared_pattern_count = len(combined_patterns_df)
if MIN_SHARED_PATTERN_VERTICAL_SUPPORT is not None:
    combined_patterns_df = combined_patterns_df[
        combined_patterns_df["vertical_support"].astype(float) >= float(MIN_SHARED_PATTERN_VERTICAL_SUPPORT)
    ].copy().reset_index(drop=True)
filtered_patterns_path = OUTPUT_DIR / f"combined_deduped_patterns_mvs_{MIN_SHARED_PATTERN_VERTICAL_SUPPORT:.2f}.csv"
combined_patterns_df.to_csv(filtered_patterns_path, index=False)
print(f"[patterns] Wrote filtered pattern bank to {filtered_patterns_path}")

shared_pattern_count = len(combined_patterns_df)
print(
    "[patterns] Shared patterns after MVS filter:",
    shared_pattern_count,
    "of",
    all_shared_pattern_count,
    f"(min vertical_support >= {MIN_SHARED_PATTERN_VERTICAL_SUPPORT:.2f})",
)
keep_tirp_feature_columns = feature_columns_for_patterns(combined_patterns_df)
print("[patterns] TIRP feature columns kept after MVS filter:", len(keep_tirp_feature_columns))

del combined_patterns_merged, combined_patterns_df

SHARED_TIRP_CHUNK_DIR = OUTPUT_DIR / "shared_tirp_chunks"
chunk_files = sorted(SHARED_TIRP_CHUNK_DIR.glob("shared_tirp_chunk_*.pkl"))
print(f"[cache] Using existing shared TIRP chunks from {SHARED_TIRP_CHUNK_DIR}: {len(chunk_files)} files")

for _name in ("observation_temporal", "raw_temporal", "feature_temporal"):
    globals().pop(_name, None)

gc.collect()

summary_metrics, metrics_by_outcome, top_features = run_multilabel_pipeline(
    OUTCOMES_TO_RUN,
    SHARED_TIRP_CHUNK_DIR,
    shared_pattern_count,
    keep_tirp_feature_columns,
)

metrics_by_outcome.to_csv(OUTPUT_DIR / "all_outcome_metrics.csv", index=False)
display(pd.Series(summary_metrics).to_frame("value"))

del labels, obs_counts, shared_pattern_count, keep_tirp_feature_columns

gc.collect()


[patterns] Loading cached per-outcome pattern files
[patterns] Loading cached patterns for DISGLYCEMIA_EVENT_Hyperglycemia
[patterns] Loading cached patterns for DISGLYCEMIA_EVENT_Hypoglycemia
[patterns] Loading cached patterns for HYPEROSMOLALITY_EVENT
[patterns] Loading cached patterns for CARDIO-VASCULAR_DISORDER_EVENT
[patterns] Loading cached patterns for KIDNEY_COMPLICATION_EVENT
[patterns] Loading cached patterns for DEATH_EVENT
[patterns] Wrote combined pattern bank to data\output\ml_pipeline\combined_deduped_patterns.csv
[patterns] Wrote filtered pattern bank to data\output\ml_pipeline\combined_deduped_patterns_mvs_0.20.csv
[patterns] Shared patterns after MVS filter: 29757 of 207439 (min vertical_support >= 0.20)
[patterns] TIRP feature columns kept after MVS filter: 35982
[cache] Using existing shared TIRP chunks from data\output\ml_pipeline\shared_tirp_chunks: 58 files

=== Multi-label model ===
[setup] Output directory: data\output\ml_pipeline\multilabel_model
[split] Crea

,positives
DISGLYCEMIA_EVENT_Hyperglycemia,11284
DISGLYCEMIA_EVENT_Hypoglycemia,2402
HYPEROSMOLALITY_EVENT,12690
CARDIO-VASCULAR_DISORDER_EVENT,892
KIDNEY_COMPLICATION_EVENT,10272
DEATH_EVENT,5431


[split] Test positives by outcome:


,positives
DISGLYCEMIA_EVENT_Hyperglycemia,3760
DISGLYCEMIA_EVENT_Hypoglycemia,800
HYPEROSMOLALITY_EVENT,4228
CARDIO-VASCULAR_DISORDER_EVENT,293
KIDNEY_COMPLICATION_EVENT,3425
DEATH_EVENT,1813


[features] Loading TIRP features from data\output\ml_pipeline\shared_tirp_chunks
[cache] Building filtered TIRP cache from dense chunks in data\output\ml_pipeline\shared_tirp_chunks
[cache] Destination: data\output\ml_pipeline\filtered_tirp_cache_mvs_0_20
[cache] Filtered columns=35982, patients=57078, dtype=float32
[cache] shared_tirp_chunk_0000.pkl: kept_columns=35612, nnz=8821404
[cache] shared_tirp_chunk_0001.pkl: kept_columns=35612, nnz=9147962
[cache] shared_tirp_chunk_0002.pkl: kept_columns=35582, nnz=9014908
[cache] shared_tirp_chunk_0003.pkl: kept_columns=35634, nnz=9111212
[cache] shared_tirp_chunk_0004.pkl: kept_columns=35624, nnz=8916174
[cache] shared_tirp_chunk_0005.pkl: kept_columns=35642, nnz=9153194
[cache] shared_tirp_chunk_0006.pkl: kept_columns=35598, nnz=8893666
[cache] shared_tirp_chunk_0007.pkl: kept_columns=35602, nnz=9160788
[cache] shared_tirp_chunk_0008.pkl: kept_columns=35612, nnz=9005024
[cache] shared_tirp_chunk_0009.pkl: kept_columns=35612, nnz=8959932
[c

,n,positives,controls,pr_auc,roc_auc,f1_at_0_5,best_f1,best_f1_threshold,outcome
0,14270,3760,10510,0.743667,0.869655,0.680504,0.682691,0.443809,DISGLYCEMIA_EVENT_Hyperglycemia
1,14270,800,13470,0.191091,0.750171,0.253953,0.266049,0.771156,DISGLYCEMIA_EVENT_Hypoglycemia
2,14270,4228,10042,0.700623,0.818905,0.645521,0.646094,0.503696,HYPEROSMOLALITY_EVENT
3,14270,293,13977,0.334262,0.900695,0.404692,0.428325,0.690181,CARDIO-VASCULAR_DISORDER_EVENT
4,14270,3425,10845,0.693045,0.828804,0.620918,0.625205,0.637992,KIDNEY_COMPLICATION_EVENT
5,14270,1813,12457,0.489054,0.825702,0.496252,0.498415,0.510819,DEATH_EVENT


,value
model,one_vs_rest_logistic_regression
n_test,14270
n_labels,6
micro_pr_auc,0.642899
macro_pr_auc,0.52529
micro_f1_at_0_5,0.596754
macro_f1_at_0_5,0.516973
micro_roc_auc,0.866181
macro_roc_auc,0.832322
n_train,42808


2320727